In [ ]:
import sys
sys.path.append("..")

from ERA_Distribution_Classes_Python.Classes.ERADist import ERADist
from ERA_Distribution_Classes_Python.Classes.ERANataf import ERANataf
from ERA_Distribution_Classes_Python.Classes.FORM_HLRF import FORM_HLRF
from ERA_Distribution_Classes_Python.Classes.FORM_fmincon import FORM_fmincon
from ERA_Distribution_Classes_Python.Classes.SuS import SuS

In [ ]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt 
from dsm_core.structure import Structure
from dsm_core.solver_2nd import Solver2ndOrder
from measures_of_nonlinearity import kappa_1, kappa_2, kappa_12, r1, r2
from syst_1_model_functions import t_S_nonlinear

## Material Properties

In [ ]:
# modified IPE 120 -> eta = 100% for linear hyperplane
E = 210e6 # kN/m2 
A = 1.321e-3 # m2
I = 2.7784576742780014e-06 # m4
h = 0.12  # m
z = h/2  # m
alpha = 1.14

In [4]:
def t_R(M_k, I=I, z=z, alpha=alpha):
    """
    Takes in the Steel Bending Strength M_k (random variable) in kN/cm2
    I in m^4
    z in m
    alpha is the plastic ratio 

    Returns the characteristic Bending Moment Resistance M_c_Rk for the given system in kNm
    """
    return I/z * M_k * 100e2 * alpha

## System Definition

In [5]:
# Vectorized Version of the Structural response function (Better for array handling later)
t_S_vectorized = np.vectorize(t_S_nonlinear, otypes=[float])

## Target characteristic values for calibrating random variables

In [ ]:
s_k = 1.1 # snow load on ground kN/m2
q_b = 0.65 # wind pressure kN/m2
w_k = q_b * 0.8 # wind load kN/m2 with c_pe,10 = 0.8 (Area D)
m_k = 35.5  # material strength kN/cm2

In [8]:
# Design Opt 1
print(t_S_nonlinear(l_1= s_k*1.5, l_2= q_b*1.5) * 1.0 / t_R(m_k))

# Design Opt 2
e_d = max(1.5 * t_S_nonlinear(l_1= s_k, l_2= q_b), 1.5 * t_S_nonlinear(l_1= s_k, l_2= q_b))
print(e_d * 1.0 / t_R(m_k))

1.1276524174800486
1.0801814472354907


## t_S_nonlinear results for calibrating t_S_hyperplane_linear

In [9]:
print(f"t_S_(l_1k, 0) = {t_S_nonlinear(l_1=s_k, l_2=0.0)}")
print(f"t_S_(0, l_2k) = {t_S_nonlinear(l_1=0.0, l_2=q_b)}")

t_S_(l_1k, 0) = 1.521903900412326
t_S_(0, l_2k) = 10.971906842505982


## Random Variables

In [10]:
# Snow time-invariant part
mu_Theta_1 = 0.81
cov_Theta_1 = 0.26
sig_Theta_1 = mu_Theta_1 * cov_Theta_1
Theta_L1 = ERADist('lognormal','MOM',[mu_Theta_1, sig_Theta_1])

# Snow load on ground
mu_L1 = 1.0
cov_L1 = 0.2
sig_L1 = mu_L1 * cov_L1
L1 = ERADist('gumbel','MOM',[mu_L1,sig_L1])

In [11]:
percentile_L1 = L1.icdf(0.98)
print(f"Snow 98% Percentile: {percentile_L1}")

Snow 98% Percentile: 1.5184551765313796


In [12]:
# Wind time-invariant part
mu_Theta_2 = 0.97
cov_Theta_2 = 0.26
sig_Theta_2 = mu_Theta_2 * cov_Theta_2
Theta_L2 = ERADist('lognormal','MOM',[mu_Theta_2, sig_Theta_2])

# Wind velocity pressure
mu_L2 = 1.0 
cov_L2 = 0.14
sig_L2 = mu_L2 * cov_L2
L2 = ERADist('gumbel','MOM',[mu_L2, sig_L2])

In [13]:
percentile_L2 = L2.icdf(0.98)
print(f"Wind 98% Percentile: {percentile_L2}")

Wind 98% Percentile: 1.3629186235719657


In [14]:
# Structural Response Model Uncertainty (from JCSS Probabilistic Model Code, Part 3, Table 3.9.1)
mu_Theta_S = 1.0
cov_Theta_S = 0.1
sig_Theta_S = mu_Theta_S * cov_Theta_S
Theta_S = ERADist('lognormal','MOM',[mu_Theta_S, sig_Theta_S])  # Distribution for Moments in frames

In [15]:
# Steel bending model uncertainty
mu_Theta_M = 1.15
cov_Theta_M = 0.05
sig_Theta_M = mu_Theta_M * cov_Theta_M
Theta_M = ERADist('lognormal','MOM',[mu_Theta_M, sig_Theta_M])

# Steel yielding strength
mu_M = 1.0
cov_M = 0.05
sig_M = mu_M * cov_M
M = ERADist('lognormal','MOM',[mu_M, sig_M])

In [16]:
percentile_M = M.icdf(0.05)
print(f"Steel 5% Percentile: {percentile_M}")

Steel 5% Percentile: 0.9199464756612658


## Shifting / Scaling Random Variables

In [17]:
# Snow Load on Ground, shifted to characteristic value
snow_shift = s_k / percentile_L1 # ratio of target to current percentile, by which mean and std get multiplied

mu_L1_shifted = mu_L1 * snow_shift
sig_L1_shifted = sig_L1 * snow_shift
L1_shifted = ERADist('gumbel','MOM',[mu_L1_shifted, sig_L1_shifted])

print(f"""Snow Load on Ground gets shifted by {snow_shift}""")
print(f"""Old mean: {mu_L1}; New mean: {mu_L1_shifted}""")
print(f"""Old std: {sig_L1}; New std: {sig_L1_shifted}""")
print(f"""Old 98th percentile: {L1.icdf(.98)}; New 98th percentile: {L1_shifted.icdf(.98)}""")
print(f"""Old COV: {L1.std()/L1.mean()}; New COV: {L1_shifted.std()/L1_shifted.mean()}""")

Snow Load on Ground gets shifted by 0.7244204616646898
Old mean: 1.0; New mean: 0.7244204616646898
Old std: 0.2; New std: 0.14488409233293795
Old 98th percentile: 1.5184551765313796; New 98th percentile: 1.0999999999999999
Old COV: 0.19999999999999998; New COV: 0.19999999999999996


In [18]:
# Wind velocity pressure, shifted to characteristic value
wind_shift = q_b / percentile_L2

mu_L2_shifted = mu_L2 * wind_shift
sig_L2_shifted = sig_L2 * wind_shift
L2_shifted = ERADist('gumbel','MOM',[mu_L2_shifted, sig_L2_shifted])

print(f"""Wind velocity pressure gets shifted by {wind_shift}""")
print(f"""Old mean: {mu_L2}; New mean: {mu_L2_shifted}""")
print(f"""Old std: {sig_L2}; New std: {sig_L2_shifted}""")
print(f"""Old 98th percentile: {L2.icdf(.98)}; New 98th percentile: {L2_shifted.icdf(.98)}""")
print(f"""Old COV: {L2.std()/L2.mean()}; New COV: {L2_shifted.std()/L2_shifted.mean()}""")

Wind velocity pressure gets shifted by 0.4769176888173018
Old mean: 1.0; New mean: 0.4769176888173018
Old std: 0.14; New std: 0.06676847643442226
Old 98th percentile: 1.3629186235719657; New 98th percentile: 0.65
Old COV: 0.14; New COV: 0.13999999999999999


In [19]:
# Steel bending resistance, shifted to characteristic value
steel_shift = m_k / percentile_M

mu_M_shifted = mu_M * steel_shift
sig_M_shifted = sig_M * steel_shift
M_shifted = ERADist('lognormal','MOM',[mu_M_shifted, sig_M_shifted])

print(f"""Steel bending resistance gets shifted by {steel_shift}""")
print(f"""Old mean: {mu_M}; New mean: {mu_M_shifted}""")
print(f"""Old std: {sig_M}; New std: {sig_M_shifted}""")
print(f"""Old 5th percentile: {M.icdf(0.05)}; New 5th percentile: {M_shifted.icdf(0.05)}""")
print(f"""Old COV: {M.std()/M.mean()}; New COV: {M_shifted.std()/M_shifted.mean()}""")

Steel bending resistance gets shifted by 38.58920158858403
Old mean: 1.0; New mean: 38.58920158858403
Old std: 0.05; New std: 1.9294600794292016
Old 5th percentile: 0.9199464756612658; New 5th percentile: 35.5
Old COV: 0.04999999999999947; New COV: 0.04999999999999946


## Characteristic values derived from Random Variables for further calculation

In [23]:
l_1k = L1_shifted.icdf(0.98)
l_2k = L2_shifted.icdf(0.98)
m_k = M_shifted.icdf(0.05)
print(f"l_1k = {l_1k:.4f} kN/m^2")
print(f"l_2k = {l_2k:.4f} kN/m^2")
print(f"m_k = {m_k:.4f} kN/cm^2")

l_1k = 1.1000 kN/m^2
l_2k = 0.6500 kN/m^2
m_k = 35.5000 kN/cm^2


## Partial Safety Factors

In [24]:
gamma_M = 1.0  # Resistance
gamma_F1 = 1.5   # Snow Load
gamma_F2 = 1.5   # Wind Load
#psi_0 = 1.0 # 0.6     # Windload

## Design Values

In [25]:
l_1d = s_k * gamma_F1
l_2d = q_b * gamma_F2 
m_d = m_k / gamma_M
print(f"l_1d = {l_1d:.4f} kN/m^2")
print(f"l_2d = {l_2d:.4f} kN/m^2")
print(f"m_d = {m_d:.4f} kN/cm^2")

l_1d = 1.6500 kN/m^2
l_2d = 0.9750 kN/m^2
m_d = 35.5000 kN/cm^2


## Measures of Nonlinearity

In [26]:
## Measures of Nonlinearity
k1 = kappa_1(l_1k=l_1k, l_1d=l_1d, t_S=t_S_nonlinear)
k2 = kappa_2(l_2k=l_2k, l_2d=l_2d, t_S=t_S_nonlinear)
k12 = kappa_12(l_1k=l_1k, l_1d=l_1d, l_2k=l_2k, l_2d=l_2d, t_S=t_S_nonlinear)
r1 = r1(l_1k=l_1k, l_2k=l_2k, t_S=t_S_nonlinear)
r2 = r2(l_1k=l_1k, l_2k=l_2k, t_S=t_S_nonlinear)


print(f"kappa1 = {k1}")
print(f"kappa2 = {k2}")
print(f"kappa12 = {k12}")
print(f"r1 = {r1}")
print(f"r2 = {r2}")

kappa1 = 1.6556150935902205
kappa2 = 0.9497325520989162
kappa12 = 1.1318416559534055
r1 = 0.11277063763196107
r2 = 0.8130006962536702


## Design parameters p for option 1 and 2 

In [27]:
# Design Opt 1
e_d_1 = t_S_nonlinear(l_1d, l_2d)

# Design Opt 2
argument_1 = gamma_F1 * t_S_nonlinear(l_1k,  (gamma_F2 / gamma_F1) * l_2k)
argument_2 = gamma_F2 * t_S_nonlinear((gamma_F1 / gamma_F2) * l_1k, l_2k)
e_d_2 = max(argument_1, argument_2)

p_opt1 = gamma_M * e_d_1 / t_R(M_k=m_d)
p_opt2 = gamma_M * e_d_2 / t_R(M_k=m_d)

print("Design Option 1:")
print(f"e_d = {e_d_1} kNm")
print(f"p_opt1 = {p_opt1}")
print(f"\n")
print("Design Option 2:")
print(f"e_d = {e_d_2} kNm")
print(f"p_opt2 = {p_opt2}")

Design Option 1:
e_d = 21.13299229197635 kNm
p_opt1 = 1.1276524174800486


Design Option 2:
e_d = 20.24335322170971 kNm
p_opt2 = 1.0801814472354907


## Construction of the Nataf Distribution

In [28]:
# Array of marginal distributions
marginal_dist_with_model_uncertainties = [Theta_M, M_shifted, Theta_L1, L1_shifted, Theta_L2, L2_shifted, Theta_S]

# Correlation matrix (no correlation yet)
dimensions = len(marginal_dist_with_model_uncertainties)
R_xx = np.eye(dimensions)

# Construction of the Nataf Distribution
nataf_with_model_uncertainties = ERANataf(M=marginal_dist_with_model_uncertainties, Correlation=R_xx)

In [29]:
# Array of marginal distributions
marginal_dist_without_model_uncertainties = [M_shifted, L1_shifted, L2_shifted]

# Correlation matrix (no correlation yet)
dimensions = len(marginal_dist_without_model_uncertainties)
R_xx = np.eye(dimensions)

# Construction of the Nataf Distribution
nataf_without_model_uncertainties = ERANataf(M=marginal_dist_without_model_uncertainties, Correlation=R_xx)

## Subset Simulation with model uncertainties

### Design Option 1

In [30]:
# deterministic design action effect
e_d_opt1 = t_S_nonlinear(l_1=gamma_F1 * s_k, l_2=gamma_F2 * q_b) # kNm

In [31]:
def g_opt_1_sus(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k) # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt1 / r_k) * x[:,0] * t_R(M_k=x[:,1])
    action_side = x[:,6] * t_S_vectorized(l_1=(x[:,2] * x[:,3]), l_2=(x[:,4] * x[:,5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [ ]:
# # %% Samples Return
# samples_return = 1
# # %% subset simulation
# N  = 10000        # Total number of samples for each level
# p0 = 0.1         # Probability of each subset, chosen adaptively

# print('\n\nSUBSET SIMULATION: ')
# [Pf_SuS_1, delta_SuS, b, Pf, b_sus, pf_sus, samplesU, samplesX, fs_iid] = SuS(N, p0, g_opt_1_sus, nataf_with_model_uncertainties, samples_return)



SUBSET SIMULATION: 
Evaluating performance function:	OK!

-Threshold intermediate level  0  =  13.36182905640665
	*aCS lambda = 0.7587935819348934 	*aCS sigma = 0.7587935819348934 	*aCS accrate = 0.44466891133557795

-Threshold intermediate level  1  =  8.833116775665184
	*aCS lambda = 0.5459096879829535 	*aCS sigma = 0.5459096879829535 	*aCS accrate = 0.43737373737373736

-Threshold intermediate level  2  =  4.313752139452495
	*aCS lambda = 0.43003396060559185 	*aCS sigma = 0.43003396060559185 	*aCS accrate = 0.43524130190796856

-Threshold intermediate level  3  =  0.0
	*aCS lambda = 0.36699868864726837 	*aCS sigma = 0.36699868864726837 	*aCS accrate = 0.43512250161186333


In [ ]:
# print("Subset Simulation for Design Option 1")
# print(f"P(F) = {Pf_SuS_1}")
# X = sp.stats.Normal()
# beta = - X.icdf(Pf_SuS_1)
# print(f"beta = {beta}")
# print(samplesX)

Subset Simulation for Design Option 1
P(F) = 0.00010500000000000002
beta = 3.7066727633500083
[array([[ 1.03991575, 37.46143772,  0.57342682, ...,  2.09510762,
         0.7159232 ,  1.07964094],
       [ 1.03991575, 37.46143772,  0.57342682, ...,  2.09510762,
         0.7159232 ,  1.07964094],
       [ 1.03991575, 37.46143772,  0.57342682, ...,  2.09510762,
         0.7159232 ,  1.07964094],
       ...,
       [ 1.16921923, 35.42998775,  0.63503104, ...,  1.89123011,
         0.55389675,  1.40144468],
       [ 1.16921923, 35.42998775,  0.63503104, ...,  1.89123011,
         0.55389675,  1.40144468],
       [ 1.16390658, 34.89451856,  0.75862349, ...,  1.94849712,
         0.60282708,  1.29682863]], shape=(10000, 7))]


---

### Design Option 2

In [34]:
# deterministic design action effect
argument_1 = gamma_F1 * t_S_nonlinear(l_1= s_k, l_2= (gamma_F2 / gamma_F1) * q_b)
argument_2 = gamma_F2 * t_S_nonlinear(l_1= (gamma_F1 / gamma_F2) * s_k, l_2= q_b)
e_d_opt2 = max(argument_1, argument_2) # kNm

In [35]:
def g_opt_2_sus(x):
    """
    LSF for Design Option 2
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt2 / r_k) * x[:,0] * t_R(M_k=x[:,1])
    action_side = x[:,6] * t_S_vectorized(l_1=(x[:,2] * x[:,3]), l_2=(x[:,4] * x[:,5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [ ]:
# # %% Samples Return
# samples_return = 1
# # %% subset simulation
# N  = 10000        # Total number of samples for each level
# p0 = 0.1         # Probability of each subset, chosen adaptively

# print('\n\nSUBSET SIMULATION: ')
# [Pf_SuS_2, delta_SuS, b, Pf, b_sus, pf_sus, samplesU, samplesX, fs_iid] = SuS(N, p0, g_opt_2_sus, nataf_with_model_uncertainties, samples_return)



SUBSET SIMULATION: 
Evaluating performance function:	OK!

-Threshold intermediate level  0  =  12.20332684773443
	*aCS lambda = 0.7108383737070808 	*aCS sigma = 0.7108383737070808 	*aCS accrate = 0.44029180695847364

-Threshold intermediate level  1  =  7.739072747035736
	*aCS lambda = 0.5241545126080532 	*aCS sigma = 0.5241545126080532 	*aCS accrate = 0.4337822671156004

-Threshold intermediate level  2  =  3.5639282926787295
	*aCS lambda = 0.4393186452006124 	*aCS sigma = 0.4393186452006124 	*aCS accrate = 0.43894500561167227

-Threshold intermediate level  3  =  0.0
	*aCS lambda = 0.38562059377387564 	*aCS sigma = 0.38562059377387564 	*aCS accrate = 0.4377777777777778


In [ ]:
# print("Subset Simulation for Design Option 2")
# print(f"P(F) = {Pf_SuS_2}")

# X = sp.stats.Normal()
# beta = - X.icdf(Pf_SuS_2)
# print(f"beta = {beta}")
# print(samplesX)

Subset Simulation for Design Option 2
P(F) = 0.00014710000000000005
beta = 3.620354196883511
[array([[ 1.17465806, 35.24122544,  0.98239561, ...,  2.12380641,
         0.6927824 ,  0.99733739],
       [ 1.19767413, 34.22723862,  1.04623981, ...,  2.2527167 ,
         0.60735485,  0.99328492],
       [ 1.22922521, 34.48799245,  0.94673825, ...,  2.33549281,
         0.62051152,  1.03747299],
       ...,
       [ 1.10492415, 37.18609638,  0.68438874, ...,  1.49873799,
         0.64930487,  1.40601819],
       [ 1.13057142, 36.88819655,  0.66617265, ...,  1.36827575,
         0.72129881,  1.43060972],
       [ 1.11976196, 36.55005517,  0.68227349, ...,  1.5034085 ,
         0.69680029,  1.47611853]], shape=(10000, 7))]


## Subset Simulation without model uncertainties

### Design Option 1

In [38]:
# deterministic design action effect
e_d_opt1 = t_S_nonlinear(l_1=gamma_F1 * s_k, l_2=gamma_F2 * q_b) # kNm

In [39]:
def g_opt_1_sus(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: M = Steel yield strength
    x[1]: L1 = Snow Load on Ground
    x[2]: L2 = Wind velocity pressure
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt1 / r_k) * t_R(M_k=x[:,0])
    action_side = t_S_vectorized(l_1=(x[:,1]), l_2=(x[:,2]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [ ]:
# # %% Samples Return
# samples_return = 1
# # %% subset simulation
# N  = 10000        # Total number of samples for each level
# p0 = 0.1         # Probability of each subset, chosen adaptively

# print('\n\nSUBSET SIMULATION: ')
# [Pf_SuS_1, delta_SuS, b, Pf, b_sus, pf_sus, samplesU, samplesX, fs_iid] = SuS(N, p0, g_opt_1_sus, nataf_without_model_uncertainties, samples_return)



SUBSET SIMULATION: 
Evaluating performance function:	OK!

-Threshold intermediate level  0  =  11.35926674922065
	*aCS lambda = 0.7443396010375672 	*aCS sigma = 0.7443396010375672 	*aCS accrate = 0.44388327721661053

-Threshold intermediate level  1  =  9.016816999623584
	*aCS lambda = 0.5288088767728638 	*aCS sigma = 0.5288088767728638 	*aCS accrate = 0.43333333333333335

-Threshold intermediate level  2  =  6.929116215024846
	*aCS lambda = 0.42785286989569477 	*aCS sigma = 0.42785286989569477 	*aCS accrate = 0.4388327721661055

-Threshold intermediate level  3  =  4.870628003721479
	*aCS lambda = 0.37622669577952034 	*aCS sigma = 0.37622669577952034 	*aCS accrate = 0.43827160493827166

-Threshold intermediate level  4  =  2.857714370526902
	*aCS lambda = 0.32883265363792474 	*aCS sigma = 0.32883265363792474 	*aCS accrate = 0.43569023569023574

-Threshold intermediate level  5  =  0.8913101817129245
	*aCS lambda = 0.30020119176550103 	*aCS sigma = 0.30020119176550103 	*aCS accrate =

In [ ]:
# print("Subset Simulation for Design Option 1")
# print(f"P(F) = {Pf_SuS_1}")
# X = sp.stats.Normal()
# beta = - X.icdf(Pf_SuS_1)
# print(f"beta = {beta}")
# print(samplesX)

Subset Simulation for Design Option 1
P(F) = 3.554000000000001e-07
beta = 4.958389358186654
[array([[35.82190254,  1.2608509 ,  1.10685141],
       [35.96805543,  1.15586561,  1.18774238],
       [35.90906345,  1.16056933,  1.23816104],
       ...,
       [37.03092112,  0.61614703,  1.30799542],
       [33.59728786,  0.69057383,  1.15865271],
       [33.59728786,  0.69057383,  1.15865271]], shape=(10000, 3))]


---

### Design Option 2

In [42]:
# deterministic design action effect
argument_1 = gamma_F1 * t_S_nonlinear(l_1= s_k, l_2= (gamma_F2 / gamma_F1) * q_b)
argument_2 = gamma_F2 * t_S_nonlinear(l_1= (gamma_F1 / gamma_F2) * s_k, l_2= q_b)
e_d_opt2 = max(argument_1, argument_2) # kNm

In [43]:
def g_opt_2_sus(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: M = Steel yield strength
    x[1]: L1 = Snow Load on Ground
    x[2]: L2 = Wind velocity pressure
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt2 / r_k) * t_R(M_k=x[:,0])
    action_side = t_S_vectorized(l_1=(x[:,1]), l_2=(x[:,2]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [ ]:
# # %% Samples Return
# samples_return = 1
# # %% subset simulation
# N  = 10000        # Total number of samples for each level
# p0 = 0.1         # Probability of each subset, chosen adaptively

# print('\n\nSUBSET SIMULATION: ')
# [Pf_SuS_2, delta_SuS, b, Pf, b_sus, pf_sus, samplesU, samplesX, fs_iid] = SuS(N, p0, g_opt_2_sus, nataf_without_model_uncertainties, samples_return)



SUBSET SIMULATION: 
Evaluating performance function:	OK!

-Threshold intermediate level  0  =  10.397703682048629
	*aCS lambda = 0.7534753927222263 	*aCS sigma = 0.7534753927222263 	*aCS accrate = 0.44287317620650957

-Threshold intermediate level  1  =  8.137452245050993
	*aCS lambda = 0.5328987619018851 	*aCS sigma = 0.5328987619018851 	*aCS accrate = 0.43524130190796845

-Threshold intermediate level  2  =  6.164238168413282
	*aCS lambda = 0.4246682850795929 	*aCS sigma = 0.4246682850795929 	*aCS accrate = 0.4342312008978676

-Threshold intermediate level  3  =  4.13821935472944
	*aCS lambda = 0.3939535758712126 	*aCS sigma = 0.3939535758712126 	*aCS accrate = 0.4430976430976431

-Threshold intermediate level  4  =  2.11098396650847
	*aCS lambda = 0.338526529560601 	*aCS sigma = 0.338526529560601 	*aCS accrate = 0.43804713804713796

-Threshold intermediate level  5  =  0.16040335072416292
	*aCS lambda = 0.30773532528425307 	*aCS sigma = 0.30773532528425307 	*aCS accrate = 0.440404

In [ ]:
# print("Subset Simulation for Design Option 2")
# print(f"P(F) = {Pf_SuS_2}")

# X = sp.stats.Normal()
# beta = - X.icdf(Pf_SuS_2)
# print(f"beta = {beta}")
# print(samplesX)

Subset Simulation for Design Option 2
P(F) = 8.265000000000004e-07
beta = 4.791789844989132
[array([[37.16892136,  1.1818232 ,  1.27434539],
       [37.50267334,  1.19775148,  1.35048919],
       [35.98770206,  1.39796933,  1.12949539],
       ...,
       [38.53315839,  0.91114567,  1.24158486],
       [34.22302689,  1.27282341,  0.96419767],
       [35.91358504,  0.5544496 ,  1.23310128]], shape=(10000, 3))]


## FORM Analysis with model uncertainties

### Design Option 1

In [53]:
def g_opt_1_FORM(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k) # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt1 / r_k) * x[0] * t_R(M_k=x[1])
    action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [54]:
# Perform FORM with HLRF
# [u_star, x_star, beta, Pf, S_F1, S_F1_T] = FORM_HLRF(g=g_opt_1, dg=[], distr=nataf, sensitivity_analysis=0, u0=0)

# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_1_FORM, dg=[], distr=nataf_with_model_uncertainties, u0=0)


*scipy.optimize.minimize() with  SLSQP  Method

  21  iterations... Reliability index =  3.7883995381727313  --- Failure probability =  7.581043988332499e-05 




In [55]:
print(f"u_star = {u_star}")
print(f"x_star = {x_star}")
print(f"(alpha_2)^2 = {(u_star/beta)**2}")
print(f"beta = {beta}")
print(f"P(F) = {Pf}")
print(f"g(X*) = {g_opt_1_FORM(x_star)}")

u_star = [-0.59586005 -0.59586     0.2652756   0.21346204  2.75528519  2.12612466
  1.18908915]
x_star = [ 1.11487144 37.41043368  0.83896966  0.72945845  1.89937665  0.65932893
  1.12034657]
(alpha_2)^2 = [0.02473871 0.0247387  0.00490324 0.0031749  0.52895846 0.31496761
 0.09851838]
beta = 3.7883995381727313
P(F) = 7.581043988332499e-05
g(X*) = -6.561194254572911e-09


### Design Option 2

In [56]:
def g_opt_2_FORM(x):
    """
    LSF for Design Option 2
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt2 / r_k) * x[0] * t_R(M_k=x[1])
    action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [51]:
# Perform FORM with HLRF
# [u_star, x_star, beta, Pf, S_F1, S_F1_T] = FORM_HLRF(g=g_opt_2, dg=[], distr=nataf, sensitivity_analysis=0, u0=1)

# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_2_FORM, dg=[], distr=nataf_with_model_uncertainties)


*scipy.optimize.minimize() with  SLSQP  Method

  29  iterations... Reliability index =  3.6529917432070116  --- Failure probability =  0.00012960128459093762 




In [52]:
print(f"u_star = {u_star}")
print(f"x_star = {x_star}")
print(f"(alpha_2)^2 = {(u_star/beta)**2}")
print(f"beta = {beta}")
print(f"P(F) = {Pf}")
print(f"g(X*) = {g_opt_2_FORM(x_star)}")

u_star = [-0.5748056  -0.57480517  0.24786927  0.19723944  2.67252948  2.032333
  1.14473246]
x_star = [ 1.11604497 37.44981334  0.835243    0.72718826  1.85959767  0.64728348
  1.11540039]
(alpha_2)^2 = [0.02475966 0.02475962 0.00460413 0.00291535 0.53523885 0.30952259
 0.0981998 ]
beta = 3.6529917432070116
P(F) = 0.00012960128459093762
g(X*) = 8.397446826791111e-07


## FORM Analysis without model uncertainties

### Design Option 1

In [57]:
def g_opt_1_FORM(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: M = Steel yield strength
    x[1]: L1 = Snow Load on Ground
    x[2]: L2 = Wind velocity pressure
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt1 / r_k) * t_R(M_k=x[0])
    action_side = t_S_vectorized(l_1=(x[1]), l_2=(x[2]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [58]:
# Perform FORM with HLRF
# [u_star, x_star, beta, Pf, S_F1, S_F1_T] = FORM_HLRF(g=g_opt_1, dg=[], distr=nataf, sensitivity_analysis=0, u0=0)

# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_1_FORM, dg=[], distr=nataf_without_model_uncertainties, u0=0)


*scipy.optimize.minimize() with  SLSQP  Method

  15  iterations... Reliability index =  5.0065573280151  --- Failure probability =  2.770608170871213e-07 




In [59]:
print(f"u_star = {u_star}")
print(f"x_star = {x_star}")
print(f"(alpha_2)^2 = {(u_star/beta)**2}")
print(f"beta = {beta}")
print(f"P(F) = {Pf}")
print(f"g(X*) = {g_opt_1_FORM(x_star)}")

u_star = [-1.21671969  0.58931065  4.82057283]
x_star = [36.26764539  0.7860103   1.18350338]
(alpha_2)^2 = [0.05906126 0.01385512 0.92708363]
beta = 5.0065573280151
P(F) = 2.770608170871213e-07
g(X*) = 4.93127913614444e-07


### Design Option 2

In [60]:
def g_opt_2_FORM(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: M = Steel yield strength
    x[1]: L1 = Snow Load on Ground
    x[2]: L2 = Wind velocity pressure
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt2 / r_k) * t_R(M_k=x[0])
    action_side = t_S_vectorized(l_1=(x[1]), l_2=(x[2]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [61]:
# Perform FORM with HLRF
# [u_star, x_star, beta, Pf, S_F1, S_F1_T] = FORM_HLRF(g=g_opt_2, dg=[], distr=nataf, sensitivity_analysis=0, u0=1)

# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_2_FORM, dg=[], distr=nataf_without_model_uncertainties)


*scipy.optimize.minimize() with  SLSQP  Method

  15  iterations... Reliability index =  4.797010117599415  --- Failure probability =  8.052573485015451e-07 




In [62]:
print(f"u_star = {u_star}")
print(f"x_star = {x_star}")
print(f"(alpha_2)^2 = {(u_star/beta)**2}")
print(f"beta = {beta}")
print(f"P(F) = {Pf}")
print(f"g(X*) = {g_opt_2_FORM(x_star)}")

u_star = [-1.16162302  0.56084161  4.62032409]
x_star = [36.36763192  0.78144641  1.13224932]
(alpha_2)^2 = [0.05863935 0.01366908 0.92769157]
beta = 4.797010117599415
P(F) = 8.052573485015451e-07
g(X*) = 7.258745036153869e-07


## Optimization of additional PSF $\gamma_{new}$

Remark: due to long simulation time with SuS, only FORM is used here

In [63]:
from scipy.optimize import minimize
from scipy.optimize import brentq

In [ ]:
beta_TRG = 3.4451629852993326 # linear FORM solution, copied from syst_1_reliability_analysis_lin.ipynb

### Design Option (1) 

In [65]:
def f(gamma_new):
    def g_opt_1(x):
        """
        LSF for Design Option 1
        Input variables: 
        x[0]: Theta_M = Resistance Model Uncertainty
        x[1]: M = Steel yield strength
        x[2]: Theta_L1 = Snow Load Model Uncertainty
        x[3]: L1 = Snow Load on Ground
        x[4]: Theta_L2 = Wind Load Model Uncertainty
        x[5]: L2 = Wind velocity pressure
        x[6]: Theta_S = Structural Response Model Uncertainty
        """
        
        # deterministic characteristic resistance 
        r_k = t_R(M_k=m_k) # kNm
        
        # assembly of the LSF
        resistance_side = (gamma_M * gamma_new *e_d_opt1 / r_k) * x[0] * t_R(M_k=x[1])
        action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
        
        # print(f"resistance = {resistance_side}")
        # print(f"action = {action_side}")
        
        return resistance_side - action_side

    [u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_1, dg=[], distr=nataf_with_model_uncertainties)
    return beta - beta_TRG

In [66]:
# this is for narrowing down the lower and upper bound of where brentq should search in the input space
for g in [0.89, 0.90]:
    print(g, f(g))


*scipy.optimize.minimize() with  SLSQP  Method

  21  iterations... Reliability index =  3.420549635108556  --- Failure probability =  0.00031247365191557113 


0.89 -0.024613350190776817

*scipy.optimize.minimize() with  SLSQP  Method

  19  iterations... Reliability index =  3.4559556364403266  --- Failure probability =  0.00027417278133411534 


0.9 0.010792651140993925


Observation: root lies between $x = 0.89$ and $x = 0.90$


In [68]:
# Finding the Root (Optimization Problem)
# increased the tolerance, otherwise solver will test out infeasible high loads and crash
gamma_new_1 = brentq(f, 0.89, 0.90,xtol=1e-4, rtol=1e-6, maxiter=50)


*scipy.optimize.minimize() with  SLSQP  Method

  21  iterations... Reliability index =  3.420549635108556  --- Failure probability =  0.00031247365191557113 



*scipy.optimize.minimize() with  SLSQP  Method

  19  iterations... Reliability index =  3.4559556364403266  --- Failure probability =  0.00027417278133411534 



*scipy.optimize.minimize() with  SLSQP  Method

  23  iterations... Reliability index =  3.4452165650375193  --- Failure probability =  0.0002853010161840009 



*scipy.optimize.minimize() with  SLSQP  Method

  25  iterations... Reliability index =  3.445064440025437  --- Failure probability =  0.00028546163321563226 




In [69]:
print(f"gamma_new = {gamma_new_1}")

gamma_new = 0.8969517452592679


### Verification of $\gamma_{new}$ for Design Option (1)

In [70]:
def g_opt_1_verification(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k) # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * gamma_new_1 * e_d_opt1 / r_k) * x[0] * t_R(M_k=x[1])
    action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [71]:
# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_1_verification, dg=[], distr=nataf_with_model_uncertainties, u0=0)


*scipy.optimize.minimize() with  SLSQP  Method

  24  iterations... Reliability index =  3.4452206453307284  --- Failure probability =  0.0002852967092771529 




In [72]:
print(f"Target Reliability Index: {beta_TRG}")
print(f"Optimized Reliability Index: {beta}")
print(f"Difference: {beta - beta_TRG}")

Target Reliability Index: 3.4451629852993326
Optimized Reliability Index: 3.4452206453307284
Difference: 5.766003139573428e-05


### Design Option (2) 

In [73]:
def f(gamma_new):
    def g_opt_2(x):
        """
        LSF for Design Option 2
        Input variables: 
        x[0]: Theta_M = Resistance Model Uncertainty
        x[1]: M = Steel yield strength
        x[2]: Theta_L1 = Snow Load Model Uncertainty
        x[3]: L1 = Snow Load on Ground
        x[4]: Theta_L2 = Wind Load Model Uncertainty
        x[5]: L2 = Wind velocity pressure
        x[6]: Theta_S = Structural Response Model Uncertainty
        """
        
        # deterministic characteristic resistance 
        r_k = t_R(M_k=m_k)  # kNm
        
        # assembly of the LSF
        resistance_side = (gamma_M * gamma_new * e_d_opt2 / r_k) * x[0] * t_R(M_k=x[1])
        action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
        
        # print(f"resistance = {resistance_side}")
        # print(f"action = {action_side}")
        
        return resistance_side - action_side

    [u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_2, dg=[], distr=nataf_with_model_uncertainties)
    return beta - beta_TRG

In [75]:
# this is for narrowing down the lower and upper bound of where brentq should search in the input space
for g in [0.93, 0.94]:
    print(g, f(g))


*scipy.optimize.minimize() with  SLSQP  Method

  26  iterations... Reliability index =  3.4235707220454383  --- Failure probability =  0.0003090208378647341 


0.93 -0.021592263253894295

*scipy.optimize.minimize() with  SLSQP  Method

  18  iterations... Reliability index =  3.457469571269962  --- Failure probability =  0.0002726369084285934 


0.94 0.012306585970629502


Observation: root lies between $x = 0.93$ and $x = 0.94$.

In [76]:
# Finding the Root (Optimization Problem)
# increased the tolerance, otherwise solver will test out infeasible high loads and crash
gamma_new_2 = brentq(f, a=0.93, b=0.94 ,xtol=1e-4, rtol=1e-6, maxiter=50)


*scipy.optimize.minimize() with  SLSQP  Method

  26  iterations... Reliability index =  3.4235707220454383  --- Failure probability =  0.0003090208378647341 



*scipy.optimize.minimize() with  SLSQP  Method

  18  iterations... Reliability index =  3.457469571269962  --- Failure probability =  0.0002726369084285934 



*scipy.optimize.minimize() with  SLSQP  Method

  22  iterations... Reliability index =  3.4452440215656015  --- Failure probability =  0.00028527203592668143 



*scipy.optimize.minimize() with  SLSQP  Method

  21  iterations... Reliability index =  3.4450714470851227  --- Failure probability =  0.00028545423315387883 




In [77]:
print(f"gamma_new = {gamma_new_2}")

gamma_new = 0.9363696154140458


### Verification of $\gamma_{new}$ for Design Option (2)

In [78]:
def g_opt_2_verification(x):
    """
    LSF for Design Option 2
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * gamma_new_2 * e_d_opt2 / r_k) * x[0] * t_R(M_k=x[1])
    action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [79]:
# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_2_verification, dg=[], distr=nataf_with_model_uncertainties, u0=0)


*scipy.optimize.minimize() with  SLSQP  Method

  21  iterations... Reliability index =  3.44520907120665  --- Failure probability =  0.00028530892636921003 




In [80]:
print(f"Target Reliability Index: {beta_TRG}")
print(f"Optimized Reliability Index: {beta}")
print(f"Difference: {beta - beta_TRG}")

Target Reliability Index: 3.4451629852993326
Optimized Reliability Index: 3.44520907120665
Difference: 4.608590731747242e-05
